# MOD13A2 NDVI Data Builder

This notebook isolates the vegetation-data engineering task from the main project notebook.

Goals:
- inspect the MOD13A2 archive in the Earthdata download folder
- process one HDF file at a time so the raw archive never needs to sit in memory at once
- save clean NDVI level and daily-regime CSV caches for the strategy notebook
- generate one simple visual of the regime shifts over time


In [ ]:
from pathlib import Path
import os
import re
from datetime import datetime
import warnings

def locate_project_root():
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'data_ag_futures').exists() and (candidate / 'notebooks_ag_futures').exists():
            return candidate
    raise FileNotFoundError('Could not locate a project root containing data_ag_futures and notebooks_ag_futures.')

PROJECT_ROOT = locate_project_root()
NOTEBOOKS_ROOT = PROJECT_ROOT / 'notebooks_ag_futures'
os.environ.setdefault('MPLCONFIGDIR', str(NOTEBOOKS_ROOT / '.mplconfig'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=FutureWarning)
DATA_ROOT = PROJECT_ROOT / 'data_ag_futures'
RAW_NASA_DIR = DATA_ROOT / 'raw' / 'nasa' / 'mod13a2'
PROCESSED_DIR = DATA_ROOT / 'processed'

EARTHDATA_ROOT_CANDIDATES = [
    Path('/Users/jlaw/projects/earthdata_downloads'),
    RAW_NASA_DIR,
]
RAW_NASA_ZIP_INSTRUCTION = f'Unzip the provided NASA MOD13A2 data into {RAW_NASA_DIR} and rerun this notebook.'

NDVI_LEVEL_CACHE_PATH = PROCESSED_DIR / 'mod13a2_ndvi_level_series.csv'
NDVI_DAILY_CACHE_PATH = PROCESSED_DIR / 'mod13a2_ndvi_daily_series.csv'
NDVI_INVENTORY_CACHE_PATH = PROCESSED_DIR / 'mod13a2_inventory.csv'
NDVI_REGIME_PLOT_PATH = PROCESSED_DIR / 'mod13a2_ndvi_regime_overview.png'

REFRESH_NDVI_FROM_RAW = False
SAVE_NDVI_DIAGNOSTICS = False

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def pick_first_existing_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def parse_modis_vi_date(path):
    match = re.search(r'\.A(\d{4})(\d{3})\.', path.name)
    if not match:
        return pd.NaT
    year = int(match.group(1))
    doy = int(match.group(2))
    return pd.Timestamp(datetime.strptime(f'{year}-{doy:03d}', '%Y-%j'))


def build_ndvi_inventory(root):
    if root is None:
        return pd.DataFrame(columns=['path', 'date', 'year', 'doy', 'tile'])
    rows = []
    for path in sorted(root.rglob('MOD13A2*.hdf')):
        date = parse_modis_vi_date(path)
        tile_match = re.search(r'\.(h\d{2}v\d{2})\.', path.name)
        rows.append({
            'path': str(path),
            'date': date,
            'year': date.year if pd.notna(date) else np.nan,
            'doy': date.dayofyear if pd.notna(date) else np.nan,
            'tile': tile_match.group(1) if tile_match else None,
        })
    if not rows:
        return pd.DataFrame(columns=['path', 'date', 'year', 'doy', 'tile'])
    return (
        pd.DataFrame(rows)
        .drop_duplicates(subset=['date', 'tile', 'path'])
        .sort_values(['date', 'tile', 'path'])
        .reset_index(drop=True)
    )


def ndvi_inventory_diagnostics(inventory):
    if inventory.empty:
        return {
            'file_count': 0,
            'unique_dates': 0,
            'tiles': [],
            'median_gap_days': np.nan,
            'max_gap_days': np.nan,
        }
    unique_dates = pd.Series(sorted(pd.to_datetime(inventory['date'].dropna().unique())))
    if unique_dates.shape[0] > 1:
        gaps = unique_dates.diff().dropna().dt.days
        median_gap_days = float(gaps.median())
        max_gap_days = int(gaps.max())
    else:
        median_gap_days = np.nan
        max_gap_days = np.nan
    return {
        'file_count': int(len(inventory)),
        'unique_dates': int(unique_dates.shape[0]),
        'tiles': sorted(inventory['tile'].dropna().unique().tolist()),
        'median_gap_days': median_gap_days,
        'max_gap_days': max_gap_days,
    }


def require_pyhdf():
    try:
        from pyhdf.SD import SD, SDC
    except ImportError as exc:
        raise ImportError('pyhdf is required to read MOD13A2 HDF4 files.') from exc
    return SD, SDC


def read_mod13a2_ndvi_mean(path):
    SD, SDC = require_pyhdf()
    hdf = SD(str(path), SDC.READ)
    sds = hdf.select('1 km 16 days NDVI')
    arr = sds.get().astype('float64')
    attrs = sds.attributes()
    fill_value = attrs.get('_FillValue', attrs.get('fillvalue', -3000))
    arr[arr == fill_value] = np.nan
    valid_range = attrs.get('valid_range')
    if valid_range is not None and len(valid_range) == 2:
        arr[(arr < valid_range[0]) | (arr > valid_range[1])] = np.nan
    scale_factor = attrs.get('scale_factor', 0.0001)
    if scale_factor in [None, 0]:
        scale_factor = 0.0001
    if scale_factor > 1:
        arr = arr / scale_factor
    else:
        arr = arr * scale_factor
    return float(np.nanmean(arr))


def build_ndvi_level_series(inventory):
    records = []
    total = len(inventory)
    for idx, row in enumerate(inventory.itertuples(index=False), start=1):
        ndvi_value = read_mod13a2_ndvi_mean(Path(row.path))
        records.append({'date': row.date, 'tile': row.tile, 'ndvi': ndvi_value})
        if idx % 50 == 0 or idx == total:
            print(f'Processed {idx}/{total} MOD13A2 files')
    ndvi = (
        pd.DataFrame(records)
        .dropna()
        .groupby('date', as_index=False)['ndvi']
        .mean()
        .sort_values('date')
        .set_index('date')
    )
    return ndvi


def add_ndvi_regime_columns(ndvi_level):
    ndvi = ndvi_level.copy()
    ndvi['ndvi_roll_mean'] = ndvi['ndvi'].rolling(12, min_periods=6).mean()
    ndvi['ndvi_roll_std'] = ndvi['ndvi'].rolling(12, min_periods=6).std()
    ndvi['ndvi_z'] = (ndvi['ndvi'] - ndvi['ndvi_roll_mean']) / ndvi['ndvi_roll_std']
    ndvi['ndvi_regime'] = np.where(
        ndvi['ndvi_z'] < -1.0,
        1.0,
        np.where(ndvi['ndvi_z'] > 1.0, -1.0, 0.0),
    )
    full_index = pd.date_range(ndvi.index.min(), ndvi.index.max(), freq='D')
    ndvi_daily = ndvi.reindex(full_index).ffill()
    ndvi_daily.index.name = 'Date'
    return ndvi, ndvi_daily


In [ ]:
earthdata_root = pick_first_existing_path(EARTHDATA_ROOT_CANDIDATES)
if earthdata_root is None:
    message = (
        'NASA MOD13A2 source directory lookup failed. '
        + RAW_NASA_ZIP_INSTRUCTION
    )
    print(message)
    raise FileNotFoundError(message)

inventory = build_ndvi_inventory(earthdata_root)
inventory_diag = ndvi_inventory_diagnostics(inventory)

print(f'Earthdata root used for MOD13A2 search: {earthdata_root}')
print(f'NDVI files found: {inventory_diag["file_count"]}')
if not inventory.empty:
    print(f'Coverage from filename dates: {inventory["date"].min().date()} to {inventory["date"].max().date()}')
    print(f'Unique composite dates: {inventory_diag["unique_dates"]}')
    print(f'Tiles detected: {inventory_diag["tiles"]}')
    print(f'Median gap between observations: {inventory_diag["median_gap_days"]} days')
    print(inventory[['date', 'tile', 'path']].tail(12).to_string(index=False))
    if SAVE_NDVI_DIAGNOSTICS:
        inventory.to_csv(NDVI_INVENTORY_CACHE_PATH, index=False)
        print(f'Saved inventory to {NDVI_INVENTORY_CACHE_PATH}')
else:
    message = (
        f'No MOD13A2 HDF files were found under {earthdata_root}. '
        + RAW_NASA_ZIP_INSTRUCTION
    )
    print(message)
    raise FileNotFoundError(message)


In [ ]:
if NDVI_LEVEL_CACHE_PATH.exists() and NDVI_DAILY_CACHE_PATH.exists() and not REFRESH_NDVI_FROM_RAW:
    ndvi_level = pd.read_csv(NDVI_LEVEL_CACHE_PATH, parse_dates=['date']).set_index('date').sort_index()
    ndvi_level, rebuilt_daily = add_ndvi_regime_columns(ndvi_level[['ndvi']])
    ndvi_daily = pd.read_csv(NDVI_DAILY_CACHE_PATH, parse_dates=['Date']).set_index('Date').sort_index()
    required_daily_cols = ['ndvi', 'ndvi_roll_mean', 'ndvi_roll_std', 'ndvi_z', 'ndvi_regime']
    missing_daily_cols = [col for col in required_daily_cols if col not in ndvi_daily.columns]
    daily_range_mismatch = ndvi_daily.empty or ndvi_daily.index.min() != rebuilt_daily.index.min() or ndvi_daily.index.max() != rebuilt_daily.index.max()
    if missing_daily_cols or daily_range_mismatch:
        ndvi_daily = rebuilt_daily
        ndvi_daily.to_csv(NDVI_DAILY_CACHE_PATH, index_label='Date')
        print(f'Rebuilt NDVI daily cache at {NDVI_DAILY_CACHE_PATH}')
    print(f'Loaded cached NDVI level data from {NDVI_LEVEL_CACHE_PATH}')
    print(f'Loaded cached NDVI daily data from {NDVI_DAILY_CACHE_PATH}')
else:
    ndvi_level = build_ndvi_level_series(inventory)
    ndvi_level, ndvi_daily = add_ndvi_regime_columns(ndvi_level)
    ndvi_level[['ndvi']].to_csv(NDVI_LEVEL_CACHE_PATH, index_label='date')
    ndvi_daily.to_csv(NDVI_DAILY_CACHE_PATH, index_label='Date')
    print(f'Saved NDVI level data to {NDVI_LEVEL_CACHE_PATH}')
    print(f'Saved NDVI daily data to {NDVI_DAILY_CACHE_PATH}')

print('NDVI level rows', len(ndvi_level))
print('NDVI daily rows', len(ndvi_daily))
print('NDVI level range', ndvi_level.index.min().date(), ndvi_level.index.max().date())


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
ndvi_level['ndvi'].plot(ax=axes[0], color='forestgreen', linewidth=1.5)
axes[0].set_title('MOD13A2 regional mean NDVI by composite date')
axes[0].grid(True, alpha=0.3)

ndvi_level['ndvi_z'].plot(ax=axes[1], color='darkorange', linewidth=1.25)
axes[1].axhline(-1.0, color='gray', linestyle='--', linewidth=1)
axes[1].axhline(1.0, color='gray', linestyle='--', linewidth=1)
axes[1].set_title('NDVI z-score used for regime classification')
axes[1].grid(True, alpha=0.3)

axes[2].step(ndvi_level.index, ndvi_level['ndvi_regime'], where='post', color='slateblue')
axes[2].set_title('NDVI regime: bullish corn = 1, neutral = 0, bearish corn = -1')
axes[2].set_yticks([-1, 0, 1])
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
if SAVE_NDVI_DIAGNOSTICS:
    plt.savefig(NDVI_REGIME_PLOT_PATH, dpi=150, bbox_inches='tight')
    print(f'Saved regime plot to {NDVI_REGIME_PLOT_PATH}')
plt.show()
